In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:


# libraries
import os
import glob
import pandas as pd
import numpy as np
from collections import deque
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. ADVANCED GRAPH ENGINE
class AdvancedFashionGraphEngine:
    def __init__(self):
        self.adjacency_list = {}

    def construct_interaction_network(self, df, user_col, item_col):
        for _, row in df.iterrows():
            try:
                user_val = str(row[user_col]).strip().split('.')[0]
                item_val = str(row[item_col]).strip().split('.')[0]

                if user_val == "" or item_val == "" or user_val == "nan" or item_val == "nan":
                    continue

                user_node = f"USR_{user_val}"
                item_node = f"ITEM_{item_val}"

                if user_node not in self.adjacency_list: self.adjacency_list[user_node] = []
                if item_node not in self.adjacency_list: self.adjacency_list[item_node] = []

                self.adjacency_list[user_node].append(item_node)
                self.adjacency_list[item_node].append(user_node)
            except:
                continue

    def execute_pagerank(self, iterations=10, damping_factor=0.85):
        nodes = list(self.adjacency_list.keys())
        if not nodes: return {}
        total_nodes = len(nodes)
        rank_scores = {node: 1.0 / total_nodes for node in nodes}

        for _ in range(iterations):
            next_state_scores = {}
            for node in nodes:
                incoming_influence = 0
                for neighbor in self.adjacency_list[node]:
                    out_degree = len(self.adjacency_list[neighbor])
                    if out_degree > 0:
                        incoming_influence += rank_scores[neighbor] / out_degree
                next_state_scores[node] = ((1 - damping_factor) / total_nodes) + damping_factor * incoming_influence
            rank_scores = next_state_scores
        return rank_scores

    def execute_bfs_score(self, start_node, max_depth=3):
        if start_node not in self.adjacency_list: return 0
        visited = set([start_node])
        queue = deque([(start_node, 0)])
        connection_count = 0

        while queue:
            node, depth = queue.popleft()
            if depth > 0: connection_count += 1
            if depth < max_depth:
                for neighbor in self.adjacency_list[node]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append((neighbor, depth + 1))
        return connection_count

    def execute_dfs_score(self, start_node, max_depth=5):
        if start_node not in self.adjacency_list: return 0
        visited = set()
        stack = [(start_node, 0)]
        max_reached_depth = 0

        while stack:
            node, depth = stack.pop()
            if node not in visited:
                visited.add(node)
                if depth > max_reached_depth:
                    max_reached_depth = depth
                if depth < max_depth:
                    for neighbor in self.adjacency_list[node]:
                        if neighbor not in visited:
                            stack.append((neighbor, depth + 1))
        return max_reached_depth

# 2. FILE SCANNING WITH DETAILED VISIBILITY (PATH UPDATED TO 'csv' FOLDER IN DRIVE)
path = '/content/drive/MyDrive/csv'
all_files = glob.glob(os.path.join(path, "*.csv"))

print(f"Standard path set to {path}.")
print(f"Total {len(all_files)} files detected in folder. Processing full analysis...\n")

graph_engine = AdvancedFashionGraphEngine()

for file in all_files:
    file_name = os.path.basename(file)
    if "fashion_dataset_complete.csv" in file:
        print(f"ℹ️ [MASTER FILE] {file_name} -> Kept separate for final training.")
        continue
    try:
        try:
            df_temp = pd.read_csv(file, encoding='utf-8')
        except:
            df_temp = pd.read_csv(file, encoding='latin-1')

        user_c = [c for c in df_temp.columns if 'user' in c.lower() or 'id' in c.lower()]
        item_c = [c for c in df_temp.columns if 'clothing' in c.lower() or 'product' in c.lower() or 'item' in c.lower()]

        if user_c and item_c:
            graph_engine.construct_interaction_network(df_temp, user_c[0], item_c[0])
            print(f"[EXTRACTED] {file_name} -> Graph connections built successfully.")
        else:
            print(f"[SKIPPED] {file_name} -> No User or Product ID mapping columns found.")
    except Exception as e:
        print(f" [ERROR SCREENING] {file_name} -> Could not parse data: {e}")

print("\n Running PageRank Algorithmic Convergence...")
network_ranks = graph_engine.execute_pagerank()
print("Global Node Rank Features Extracted!")

# 3. MAPPING AND ENCODING TABULAR / NUMERIC FEATURES
print("\n Mapping graph features onto fashion_dataset_complete.csv...")
main_file = os.path.join(path, "fashion_dataset_complete.csv")

try:
    df_master = pd.read_csv(main_file, encoding='utf-8')
except:
    df_master = pd.read_csv(main_file, encoding='latin-1')

def safe_node_string(val):
    if pd.isnull(val):
        return ""
    return str(val).strip().split('.')[0]

df_master['clean_item_node'] = df_master['clothing_type'].apply(safe_node_string)

df_master['pagerank_score'] = df_master['clean_item_node'].apply(
    lambda x: network_ranks.get(f"ITEM_{x}", 0.0) if x != "" else 0.0
)

print("Processing Breadth-First-Search Features...")
df_master['bfs_density_score'] = df_master['clean_item_node'].apply(
    lambda x: graph_engine.execute_bfs_score(f"ITEM_{x}") if x != "" else 0
)

print("Processing Depth-First-Search Features...")
df_master['dfs_depth_score'] = df_master['clean_item_node'].apply(
    lambda x: graph_engine.execute_dfs_score(f"ITEM_{x}") if x != "" else 0
)

df_master['daa_graph_popularity_score'] = (
    df_master['pagerank_score'] +
    df_master['bfs_density_score'] * 0.01 +
    df_master['dfs_depth_score'] * 0.1
)

feature_cols = [
    'gender', 'age_group', 'body_type', 'skin_tone', 'height',
    'style_preference', 'budget_level', 'fabric', 'primary_color',
    'pattern', 'occasion', 'mood', 'season', 'weather', 'daa_graph_popularity_score'
]

print("\n🔄 Compiling Mixed Alphanumeric Text and Tabular Categories to Integer matrices...")
for col in feature_cols:
    if df_master[col].dtype == 'object':
        df_master[col] = df_master[col].astype(str).str.strip()
        df_master[col] = df_master[col].astype('category').cat.codes

X = df_master[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y = pd.to_numeric(df_master['purchased'], errors='coerce').fillna(0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. RANDOM FOREST ENGINE TRAINING
print("\n Feeding Hybrid Tabular & Numeric Matrices into Random Forest Engine...")
model = RandomForestClassifier(n_estimators=150, max_depth=18, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test) * 100
print(f"\n BOOM! SYSTEM TRAINED SUCCESSFULLY WITH 15 COMPREHENSIVE FEATURES!")
print(f"Final Classification Accuracy: {accuracy:.2f}%")

model_filename = 'fashion_hybrid_model.pkl'
joblib.dump(model, model_filename)
print(f" File Saved! Download '{model_filename}' from the left side panel files now!")

Standard path set to /content/drive/MyDrive/csv.
Total 17 files detected in folder. Processing full analysis...

[EXTRACTED] garments_worker_productivity.csv -> Graph connections built successfully.
[SKIPPED] Countries-with-Top-Sellers-(Fashion-C2C).csv -> No User or Product ID mapping columns found.
[SKIPPED] Buyers-repartition-by-country.csv -> No User or Product ID mapping columns found.
[EXTRACTED] 6M-0K-99K.users.dataset.public.csv -> Graph connections built successfully.
[EXTRACTED] Comparison-of-Sellers-by-Gender-and-Country.csv -> Graph connections built successfully.
[SKIPPED] ASOS_Trustpilot_45.csv -> No User or Product ID mapping columns found.
[SKIPPED] ASOS_Trustpilot_13.csv -> No User or Product ID mapping columns found.
[EXTRACTED] shein_mens_fashion.csv -> Graph connections built successfully.
[EXTRACTED] atlas_data.csv -> Graph connections built successfully.
[SKIPPED] E-commerce_winter_jacket_data_2024.csv -> No User or Product ID mapping columns found.
[SKIPPED] Fash